In [0]:
# read from bronze
from pyspark.sql import functions as F

base = "/Volumes/workspace/test_pipeline/test_pipeline_volume"

customers = spark.read.parquet(f"{base}/bronze/customers")
orders = spark.read.parquet(f"{base}/bronze/orders")

In [0]:
# clean customers
customers_clean = (
    customers
    .dropDuplicates(["customer_id"])
    .withColumn("customer_city", F.lower(F.trim("customer_city")))
    .withColumn("customer_state", F.upper(F.trim("customer_state")))
    .withColumn("customer_zip_code_prefix", F.lpad("customer_zip_code_prefix", 5, "0"))
)

In [0]:
# clean orders
# handles both "2017-10-02 10:56:00" and "02/10/17 10:56" date styles
def to_ts(c):
    return F.coalesce(
        F.expr(f"try_to_timestamp({c}, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr(f"try_to_timestamp({c}, 'dd/MM/yy H:mm')"),
    )

orders_clean = (
    orders
    .dropDuplicates(["order_id"])
    .withColumn("order_purchase_ts", to_ts("order_purchase_timestamp"))
    .withColumn("order_delivered_ts", to_ts("order_delivered_customer_date"))
    .withColumn("delivery_days", F.datediff("order_delivered_ts", "order_purchase_ts"))
    .select("order_id", "customer_id", "order_status",
            "order_purchase_ts", "order_delivered_ts", "delivery_days")
)

In [0]:
silver = orders_clean.join(customers_clean, on="customer_id", how="left")
display(silver.limit(5))

In [0]:
silver.write.mode("overwrite").parquet(f"{base}/silver/orders_customers")
print("Silver written:", silver.count())